# Push Rowan Trained Adapters To GitHub

Run this notebook from a fresh Colab runtime after your trained adapter folders have been saved in Google Drive. It does not train anything.

Expected adapter names:

- `rowan-qwen3-1.7b-sft`
- `rowan-qwen3-1.7b-reward`


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure Paths


In [ ]:
from pathlib import Path

GITHUB_REPO = 'pianomaster99/isekai'
MODEL_BRANCH = 'main'
REPO_DIR = Path('/content/isekai')
DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_DRIVE_MODEL_DIR = DRIVE_ROOT / 'isekai-rowan-models'
ADAPTER_NAMES = [
    'rowan-qwen3-1.7b-sft',
    'rowan-qwen3-1.7b-reward',
]
TARGET_MODEL_DIR = REPO_DIR / 'models'

print('repo:', GITHUB_REPO)
print('drive model folder:', PREFERRED_DRIVE_MODEL_DIR)


## 3. Find Adapter Folders In Drive


In [ ]:
def find_adapter_dir(adapter_name):
    candidates = [PREFERRED_DRIVE_MODEL_DIR / adapter_name]
    candidates.extend(
        path.parent
        for path in DRIVE_ROOT.rglob('adapter_config.json')
        if path.parent.name == adapter_name
    )
    for candidate in candidates:
        if (candidate / 'adapter_config.json').exists():
            return candidate
    return None

adapter_sources = {}
for name in ADAPTER_NAMES:
    source = find_adapter_dir(name)
    adapter_sources[name] = source
    print(name, '=>', source)

missing = [name for name, source in adapter_sources.items() if source is None]
assert not missing, 'Missing adapter folders in Drive: ' + ', '.join(missing)


## 4. Enter GitHub Token

Use a GitHub personal access token with repo write access. The token is not stored in the repo or printed in git remotes.


In [ ]:
from google.colab import userdata
import getpass

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

if not GH_TOKEN:
    GH_TOKEN = getpass.getpass('GitHub token: ')

assert GH_TOKEN, 'A GitHub token is required to push to GitHub.'


## 5. Clone Repo, Copy Adapters, Push With Git LFS


In [ ]:
import os
import shutil
import stat
import subprocess

def run(cmd, cwd=None, check=True, env=None):
    cwd = Path(cwd or Path.cwd())
    print('+', ' '.join(str(part) for part in cmd))
    return subprocess.run(cmd, cwd=str(cwd), check=check, env=env)

!apt-get update -qq && apt-get install -y -qq git-lfs
run(['git', 'lfs', 'install'], cwd=Path('/content'))

shutil.rmtree(REPO_DIR, ignore_errors=True)
run(['git', 'clone', f'https://github.com/{GITHUB_REPO}.git', str(REPO_DIR)], cwd=Path('/content'))
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

TARGET_MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, source in adapter_sources.items():
    target = TARGET_MODEL_DIR / name
    shutil.rmtree(target, ignore_errors=True)
    shutil.copytree(source, target)
    assert (target / 'adapter_config.json').exists(), f'Copy failed for {name}'
    print('copied', source, '->', target)

askpass = Path('/content/git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo x-access-token ;;\n*) echo "$GH_TOKEN" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
git_env = os.environ.copy()
git_env['GIT_ASKPASS'] = str(askpass)
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GH_TOKEN'] = GH_TOKEN

run(['git', 'config', 'user.email', 'colab-training-bot@example.com'])
run(['git', 'config', 'user.name', 'Colab Training Bot'])
run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GITHUB_REPO}.git'])
run(['git', 'checkout', MODEL_BRANCH])
run(['git', 'pull', 'origin', MODEL_BRANCH], env=git_env)
run(['git', 'lfs', 'track', 'models/rowan-qwen3-1.7b-sft/**'])
run(['git', 'lfs', 'track', 'models/rowan-qwen3-1.7b-reward/**'])
run(['git', 'lfs', 'track', '*.safetensors'])
run(['git', 'add', '.gitattributes'])
run(['git', 'add', '-f', 'models/rowan-qwen3-1.7b-sft', 'models/rowan-qwen3-1.7b-reward'])
run(['git', 'status', '--short'], check=False)
commit = subprocess.run(['git', 'commit', '-m', 'Add trained Rowan adapters'], cwd=str(REPO_DIR))
if commit.returncode != 0:
    print('No model changes to commit.')
run(['git', 'push', 'origin', MODEL_BRANCH], env=git_env)
askpass.unlink(missing_ok=True)


## 6. Verify What Was Pushed


In [ ]:
!git status --short
!git log --oneline -3
!find models/rowan-qwen3-1.7b-sft models/rowan-qwen3-1.7b-reward -maxdepth 1 -type f -printf '%p\n' | sort
